In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Matplotlib is building the font cache; this may take a moment.


In [34]:
Data = pd.read_csv('../Data/Nova_pay_combined.csv')

In [42]:
pd.set_option('display.max_columns',None)
Data.head(5)


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,device_id,new_device,ip_address,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,9f292dcc-3297-4947-a260-6a1ef69041ff,False,221.78.171.180,US,False,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,3a95b9f5-309f-4684-a46d-e2ff2435bf78,True,120.12.20.29,CA,False,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,a4737752-9aac-43ed-9d8b-2ccdffc24052,False,223.96.181.93,US,False,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,6aeb85a3-5603-4221-896c-9e6882764f1a,False,186.228.15.74,US,False,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,a5b9250e-dbe0-4c5f-a6e7-5492b7349402,False,11.82.47.62,US,False,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [44]:
Data['is_fraud'].value_counts()

is_fraud
0    10403
1      997
Name: count, dtype: int64

## This data has problem of class imbalance - SMOTE TECHNIQUES

In [48]:

Data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  str    
 1   customer_id                11400 non-null  str    
 2   timestamp                  11371 non-null  str    
 3   home_country               11400 non-null  str    
 4   source_currency            11400 non-null  str    
 5   dest_currency              11400 non-null  str    
 6   channel                    11400 non-null  str    
 7   amount_src                 11400 non-null  str    
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  str    
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  str    
 14  i

In [46]:
Data['timestamp']

0        2022-10-03 18:40:59.468549+00:00
1        2022-10-03 20:39:38.468549+00:00
2        2022-10-03 23:02:43.468549+00:00
3        2022-10-04 01:08:53.468549+00:00
4        2022-10-04 09:35:03.468549+00:00
                       ...               
11395    2025-11-25 10:05:35.573611+00:00
11396    2025-11-26 07:09:56.573611+00:00
11397    2025-11-27 06:19:11.573611+00:00
11398    2025-11-28 00:53:28.573611+00:00
11399    2025-11-29 20:10:47.573611+00:00
Name: timestamp, Length: 11400, dtype: str

In [58]:
Data.isnull().mean()*100>0
null_columns = Data.isnull().mean()*100>0
null_columns[null_columns ==True].index.tolist()

['timestamp',
 'amount_usd',
 'fee',
 'ip_address',
 'ip_country',
 'kyc_tier',
 'device_trust_score']

In [61]:
Data.describe(include = ['str','bool'])

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,device_id,new_device,ip_address,ip_country,location_mismatch,kyc_tier
count,11400,11400,11371,11400,11400,11400,11400,11400,11400,11400,11095,11099,11400,11100
unique,11200,1315,11141,7,3,9,12,9856,2113,2,10900,9,2,14
top,4662cb2d-21b2-4390-ba38-5a1eddebdc7c,402cccc9-28de-45b3-9af7-cc5302aa1f93,0000-00-00T00:00:00Z,US,USD,NGN,mobile,100.0,e70db499-19e1-4927-b04f-3ebfcf62e33c,False,194.179.17.152,US,False,standard
freq,2,1510,21,7940,8031,1474,6366,15,87,10047,2,6848,9548,7931


In [62]:
Data.describe()

,amount_usd,fee,exchange_rate_src_to_dest,ip_risk_score,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,11095.000000,11105.000000,11400.000000,11400.000000,11400.000000,11105.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000
mean,452.022083,100.309441,167.540397,0.396726,393.793158,0.653681,0.048509,0.267134,0.458333,0.723509,0.045501,0.087456
std,1403.973062,958.128504,382.023827,0.270507,342.348393,0.273012,0.256194,0.142983,1.524494,1.958390,0.084942,0.282515
min,7.230000,-1.000000,0.592000,0.004000,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,92.465000,2.380000,1.000000,0.209000,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,163.480000,3.500000,7.142857,0.325000,298.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,302.190000,5.550000,73.529412,0.487000,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,12498.570000,9999.990000,1388.888889,1.200000,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


In [65]:
Data.duplicated().sum()

np.int64(200)

In [68]:
Data['timestamp'] = pd.to_datetime(Data['timestamp'], errors='coerce')
Data['amount_src'] = pd.to_numeric(Data['amount_src'], errors ='coerce')
                                

In [70]:
Data[['amount_src','timestamp']].dtypes

amount_src                float64
timestamp     datetime64[us, UTC]
dtype: object

In [72]:
for col in Data.select_dtypes(include=['str','bool']).columns.tolist():
    print(Data[col].value_counts())
    print('--------------------------------')

transaction_id
4662cb2d-21b2-4390-ba38-5a1eddebdc7c    2
e69d481f-2ee3-4621-a58a-73fec8c63eca    2
8537371a-7eae-4108-89c9-5660e5025a88    2
2e553c23-063e-4b5e-aa2e-8a41d6aa61ca    2
a9198fb0-d831-48df-b434-c30a39078aa3    2
                                       ..
b5ef3323-46d2-4f6c-9c19-43df4e55df48    1
54f00f78-94aa-4a08-bf6b-12877aa47186    1
6a51f0e8-f5d1-4fe6-91a0-655fccd79fa5    1
4aad7389-2b62-4885-a23e-aa3ecd5cfaf9    1
fdffeb16-192a-4483-9b1e-9928e23269c2    1
Name: count, Length: 11200, dtype: int64
--------------------------------
customer_id
402cccc9-28de-45b3-9af7-cc5302aa1f93    1510
d71c91b4-fee8-4104-9856-a5c6109a62e3    1355
7041b9c1-3719-4ca8-9a6b-811b47cea6c0    1345
6d0d9b27-fa26-45f8-93b1-2df29d182d9c    1066
af8ca4c4-8703-4c55-b66c-2b76cd70040d     915
                                        ... 
8734e6a7-f34e-49f2-b559-1d62a756664c       1
c18070a6-bdb9-4df3-9df7-5d92c0bff424       1
2003e2a1-f3b1-4e2e-8d0e-488cfacf68ca       1
8b1cf558-4ed7-48ee-b330-75db6efd

## Task DuE Monday
. Dropping identifiers
. deal with duplicates
. Deal with null values
. fix home country to be consistent


In [73]:
Data[['transaction_id','customer_id', 'device_id']].drop

<bound method DataFrame.drop of                              transaction_id  \
0      fee8542d-8ee6-4b0d-9671-c294dd08ed26   
1      bfdb9fc1-27fe-4a85-b043-4d813d679259   
2      fc855034-3ea5-4993-9afa-b511d93fe5e8   
3      2cf8c08e-42ec-444d-a755-34b9a2a0a4ca   
4      d907a74d-b426-438d-97eb-dbe911aca91c   
...                                     ...   
11395  b5ef3323-46d2-4f6c-9c19-43df4e55df48   
11396  54f00f78-94aa-4a08-bf6b-12877aa47186   
11397  6a51f0e8-f5d1-4fe6-91a0-655fccd79fa5   
11398  4aad7389-2b62-4885-a23e-aa3ecd5cfaf9   
11399  fdffeb16-192a-4483-9b1e-9928e23269c2   

                                customer_id  \
0      402cccc9-28de-45b3-9af7-cc5302aa1f93   
1      67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad   
2      6d0d9b27-fa26-45f8-93b1-2df29d182d9c   
3      7bd5200c-5d19-44f0-9afe-8b339a05366b   
4      70a93d26-8e3a-4179-900c-a4a7a74d08e5   
...                                     ...   
11395  8734e6a7-f34e-49f2-b559-1d62a756664c   
11396  c18070a6-bdb9-4df3-9

In [ ]:
# drop identifier columns
cols_to_drop = ['transaction_id', 'customer_id', 'device_id']
Data.drop(columns=cols_to_drop, inplace=True)

# quick check
Data.head()